In [24]:
import pandas as pd
from pathlib import Path


from pymongo import MongoClient
from dotenv import load_dotenv
import os

In [25]:
load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")

client = MongoClient(MONGO_URI)
db=client["PolymerPrediction"]
collection_name="biceranoTg"
collection=db[collection_name]
target="Tg"
#print(collection.count_documents({}))

docs = list(collection.find({}))
df = pd.DataFrame(docs)

# espando il dizionario "properties"
prop_df = pd.json_normalize(df["properties"])

# integro nel dataframe
df = pd.concat([df.drop(columns=["properties"]), prop_df], axis=1)

# rimuovo righe senza SMILES o target 
df = df.dropna(subset=["smiles", target])
df.describe()

KeyboardInterrupt: 

In [11]:
# Cell 2: funzione per creare il CSV principale per ChemProp (data_path)

def create_chemprop_dataset(
    df: pd.DataFrame,
    smiles_col: str = "smiles",
    target_cols: list = None,
    output_path: str = f"data/{collection_name}.csv"
):
    
    # Se target_cols non è specificato, prendiamo tutte le numeriche tranne smiles_col
    if target_cols is None:
        numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
        if smiles_col in numeric_cols:
            numeric_cols.remove(smiles_col)
        target_cols = numeric_cols

    # Controlli base
    missing = [col for col in [smiles_col] + target_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Le seguenti colonne mancano nel DataFrame: {missing}")

    # Seleziona solo smiles + target
    dataset = df[[smiles_col] + target_cols].copy()

    # Rinominare la colonna degli SMILES in 'smiles' (standard ChemProp)
    if smiles_col != "smiles":
        dataset = dataset.rename(columns={smiles_col: "smiles"})

    # Salva
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    dataset.to_csv(output_path, index=False)
    print(f"Dataset ChemProp salvato in: {output_path.resolve()}")
    print("Colonne nel CSV:", dataset.columns.tolist())

    return dataset


In [12]:
# Cell 3: creazione del CSV ChemProp + comando da lanciare

target_cols = ["Tg"]

chemprop_df = create_chemprop_dataset(
    df=df,
    smiles_col="smiles",                    
    target_cols=target_cols,
    output_path=f"data/{collection_name}.csv"
)

chemprop_df.head()

Dataset ChemProp salvato in: C:\Users\Vale\OneDrive\Desktop\uni\tirocinio\polymer_prediction\GNNs\data\biceranoTg.csv
Colonne nel CSV: ['smiles', 'Tg']


,smiles,Tg
0,[*]O[Si][*](CC)CC,130
1,[*]O[Si][*](C)C,152
2,[*]C/C=C\C[*],171
3,[*]C[Si][*](C)C,173
4,[*]O[Si][*](C)c1ccccc1,187


# Cross Validation

In [23]:
import subprocess
import sys
import os

chemprop_exe = os.path.join(sys.prefix, "Scripts", "chemprop.exe")

cmd = [
    chemprop_exe, "train",
    "-i", f"data/{collection_name}.csv",      # data_path
    "-o", "results_dmppn_cv",                # output dir
    "--target-columns", target,              # colonna target
    "-t", "regression",                      # tipo di task
    "--split", "SCAFFOLD_BALANCED",          # tipo di split
    "--num-replicates", "5",                 # numero di replicati
    "-b", "16",                              # batch size
    "--epochs", "120",                       # epoche
    "--message-hidden-dim", "150",           # dimensione hidden
    "--depth", "3",                          # profondità MPNN
    "--dropout", "0.3"                       # dropout
]

# Scrivi l’output su file invece che stamparlo tutto in notebook
with open("chemprop_stdout.log", "w") as f_out, open("chemprop_stderr.log", "w") as f_err:
    result = subprocess.run(cmd, stdout=f_out, stderr=f_err, text=True)

print("Chemprop terminato con returncode:", result.returncode)
print("STDOUT completo salvato in: chemprop_stdout.log")
print("STDERR completo salvato in: chemprop_stderr.log")

Chemprop terminato con returncode: 0
STDOUT completo salvato in: chemprop_stdout.log
STDERR completo salvato in: chemprop_stderr.log


# Final Model Training

In [ ]:

from chemprop.train import run_training
from chemprop.args import TrainArgs

args_train_list = [
    '--data_path', 'data/bicerano_allprops.csv',
    '--dataset_type', 'regression',
    '--target_columns', *target_cols,
    '--save_dir', 'results_dmppn_final',
    '--split_type', 'scaffold_balanced',
    '--epochs', '150',
    '--batch_size', '16',
    '--hidden_size', '300',
    '--depth', '3',
    '--dropout', '0.3'
]

final_args = TrainArgs().parse_args(args_train_list)

run_training(final_args)

print("✔️ Modello finale salvato in results_dmppn_final")
